In [1]:
import io.github.francescodonnini.csv.CsvIssueApi
import io.github.francescodonnini.csv.CsvJavaClassApi
import io.github.francescodonnini.csv.CsvJavaMethodApi
import io.github.francescodonnini.csv.CsvReleaseApi
import org.eclipse.jgit.api.Git
import org.eclipse.jgit.revwalk.RevCommit
import org.eclipse.jgit.storage.file.FileRepositoryBuilder
import kotlin.io.path.Path

val project = "SYNCOPE"
val gitPath = "/home/francesco/Documents/Università/ISW/Progetto/60/sources/${project.toLowerCase()}/.git"
val path = "/home/francesco/Documents/Università/ISW/Progetto/60/data/$project"
val releaseApi = CsvReleaseApi(path + "/releases.csv")
val git = Git(FileRepositoryBuilder()
                .setGitDir(Path(gitPath).toFile())
                .build())
val commits = ArrayList<RevCommit>();
git.log().call().forEach { commits.add(it) }
val issueApi = CsvIssueApi(path + "/issues.csv", releaseApi.getLocal(), commits)

In [2]:
val tags = git.tagList().call()
    .map { Pair(it.name, git.getRepository().parseCommit(it.objectId).authorIdent.`when`) }
    .map { it -> Pair(it.first.replace("refs/tags/syncope-", ""), java.text.SimpleDateFormat("yyyy-MM-dd").format(it.second)) }
    .sortedBy { it -> it.second }
    .toList()
val releases = releaseApi.getLocal()
    .sortedBy { it -> it.releaseDate }

In [3]:
val issues = issueApi.getLocal()
    .sortedBy { it.created }
    .toList()

In [4]:
issues.size

614

In [6]:
val issuesFrame = dataFrameOf(
    "key" to issues.map { it.key },
    "created" to issues.map { it.created },
    "affectedVersions" to issues.map { it.affectedVersions.map { it -> if (it == null) "null" else it.name }.joinToString(",") },
    "fixVersion" to issues.map { it.fixVersion.name },
    "commits" to issues.map { it.commits.map { it -> it.name.substring(0..5) }.joinToString(",") }
)
issuesFrame


key,created,affectedVersions,fixVersion,commits
SYNCOPE-187,2012-08-14T08:35:21,"1.0.0-incubating,1.1.0",1.0.1-incubating,945a21
SYNCOPE-196,2012-09-03T10:32:43,1.0.0-incubating,1.0.2-incubating,6bfba4
SYNCOPE-197,2012-09-03T10:35:02,,1.1.0,947917
SYNCOPE-267,2013-01-08T10:00:41,"1.0.4,1.0.5,1.1.0",1.0.5,"5d58e0,415c57"
SYNCOPE-269,2013-01-10T11:08:23,"1.0.4,1.1.0",1.0.5,4fefb4
SYNCOPE-274,2013-01-15T11:35:22,1.0.4,1.0.5,72e4d9
SYNCOPE-285,2013-01-23T10:09:53,,1.2.0-M1,"23a34b,76a486,c63fa3"
SYNCOPE-294,2013-01-28T11:30:59,1.0.5,1.0.6,968bd6
SYNCOPE-297,2013-01-29T10:53:57,,1.1.0,e5daef
SYNCOPE-301,2013-01-30T11:27:32,"1.0.5,1.1.0",1.0.6,e52201


In [19]:
val all = CsvJavaClassApi(path + "/classes.csv").getLocal()
val methods = CsvJavaMethodApi(path + "/lbl-methods.csv", all)
    .getLocal()
val classMethodMapping = methods.groupBy { it.javaClass.commit }

In [20]:
val classes = methods.map { it.javaClass }.toList()
classes.size

95064

In [23]:
%use dataframe
classes.sortedBy { it.time }
val classesFrame = dataFrameOf(
    "name" to classes.map { it.name },
    "commit" to classes.map { it.commit },
    "time" to classes.map { it.time }
)
classesFrame


name,commit,time
UserDataBinder,e4d85752fa677b7255d3e011b6d5bc4d27ae2e10,2012-04-16T15:10:59
PasswordPolicyTO,851cdd64e3b4f27d542c65449d50d716c3acea3c,2012-04-18T15:52:29
ResourceController,630dc73548ce378e95acb50e122b612eecc2988f,2012-04-18T08:31:23
TaskTest,4305a1a0e41d29a0251e09eb6880cc5c4a1a4efb,2012-04-04T13:51:56
EncryptionUtil,add292219ddecb4099a01b4721c1ad6ff725bbc0,2011-04-14T13:10:44
SearchMode,34453842397fc2fa9228aca0c015a87ac5134796,2011-01-19T09:19:39
QueryUtils,a87430ff351cbaefdd20ec0afaff1a8c908b505d,2010-07-22T10:00:16
WorkflowFormPropertyTO,851cdd64e3b4f27d542c65449d50d716c3acea3c,2012-04-18T15:52:29
ResourceDAOImpl,a7e14552ab944a08142942cd51707f61c0c53c13,2010-12-02T13:05:47
PropagationTO,047fc4764ef9cd813d79103853713543600b7078,2012-03-16T15:10:26


In [26]:
import io.github.francescodonnini.model.JavaClass
import io.github.francescodonnini.model.Release
import java.time.LocalDate

val releaseClassMapping = mutableMapOf<Release, MutableList<JavaClass>>()
var prev = LocalDate.MIN
for (r in releases) {
    val curr = r.releaseDate
    classes
        .filter { it -> !it.time.toLocalDate().isBefore(prev) }
        .filter { it -> !it.time.toLocalDate().isAfter(curr) }
        .forEach { it -> releaseClassMapping.getOrPut(r) { mutableListOf<JavaClass>() }.add(it) }
    prev = curr
}

In [27]:
releaseClassMapping.entries.stream().mapToInt { (k, v) -> v.size }.average().orElse(0.0)

3271.4666666666667

In [12]:
import io.github.francescodonnini.model.Issue
import org.eclipse.jgit.diff.DiffFormatter
import org.eclipse.jgit.util.io.DisabledOutputStream

val df = DiffFormatter(DisabledOutputStream.INSTANCE)
df.setRepository(git.repository)
df.setDetectRenames(true)
val issue = issues[6]
println(issue)

Issue[affectedVersions=[(12323749, 1.0.5, 2013-01-23), (12322504, 1.1.0, 2013-04-05)], created=2013-01-30T11:27:32, fixVersion=(12324001, 1.0.6, 2013-02-27), openingVersion=(12324001, 1.0.6, 2013-02-27), commits=[commit e522014cc19b53d6dca57e6f80f2c7335a81590e 1359568607 -----sp], key=SYNCOPE-301, project=Syncope]


In [13]:
import io.github.francescodonnini.model.LineRange
import org.eclipse.jgit.diff.Edit

issue.commits.forEach { commit ->
    println("parent:\t${commit.parents[0].name}")
    println("commit:\t${commit.name}")
    df.scan(commit.parents[0].tree, commit.tree).forEach { diff ->
        println(diff)
        val path = diff.newPath
        println(path)
        val editList = df.toFileHeader(diff).toEditList()
        for (edit in editList) {
            println(edit)
            println("A\t${edit.beginA} ${edit.endA}")
            println("B\t${edit.beginB} ${edit.endB}")
            when (edit.type) {
                Edit.Type.DELETE, Edit.Type.INSERT, Edit.Type.REPLACE -> {
                    edit.extendB()
                    val range = LineRange(edit.beginB, edit.endB)
                    val touched = methods[commit.name]
                        ?.filter { it.javaClass.path.toString() == path }
                        ?.filter { it.range.intersects(range) }
                    if (touched == null || touched.isEmpty())
                        println("no method touched")
                    else
                        println("method touched: ${touched}")
                }
                else -> { println("${edit.type} not supported yet") }
            }
        }
    }
}


parent:	64739979c172c88fe836ab0b95bbf4754fd9c21b
commit:	e522014cc19b53d6dca57e6f80f2c7335a81590e
DiffEntry[MODIFY core/src/main/java/org/apache/syncope/core/persistence/beans/AbstractBaseBean.java]
core/src/main/java/org/apache/syncope/core/persistence/beans/AbstractBaseBean.java
DELETE(28-29,28-28)
A	28 29
B	28 28
no method touched
REPLACE(71-72,70-71)
A	71 72
B	70 71
method touched: [boolean isBooleanAsInteger(Integer) on 2013-01-30 17:56]
REPLACE(101-102,100-101)
A	101 102
B	100 101
method touched: [String[] getExcludeFields() on 2013-01-30 17:56]
DiffEntry[MODIFY core/src/main/java/org/apache/syncope/core/persistence/beans/user/SyncopeUser.java]
core/src/main/java/org/apache/syncope/core/persistence/beans/user/SyncopeUser.java
REPLACE(494-495,494-495)
A	494 495
B	494 495
method touched: [Boolean isSuspended() on 2013-01-30 17:56]
DiffEntry[MODIFY core/src/main/java/org/apache/syncope/core/security/SyncopeAuthenticationProvider.java]
core/src/main/java/org/apache/syncope/core/secur

In [50]:
methods[commit.name]

[AttributeTO attributeTO(String,String) on 2013-01-08 11:48, AttributeMod attributeMod(String,String) on 2013-01-08 11:48, RestTemplate anonymousRestTemplate() on 2013-01-08 11:48, void setupRestTemplate(String,String) on 2013-01-08 11:48, void resetRestTemplate() on 2013-01-08 11:48]

In [2]:
import io.github.francescodonnini.model.LineRange
import org.eclipse.jgit.api.Git
import org.eclipse.jgit.diff.DiffFormatter
import org.eclipse.jgit.diff.Edit
import org.eclipse.jgit.revwalk.RevCommit
import org.eclipse.jgit.storage.file.FileRepositoryBuilder
import org.eclipse.jgit.util.io.DisabledOutputStream
import kotlin.io.path.Path


val git = Git(
    FileRepositoryBuilder()
        .setGitDir(Path("/home/francesco/Documents/Università/ISW/git-diff-test/.git").toFile())
        .build()
)
val commits = ArrayList<RevCommit>();
git.log().call().forEach { commits.add(it) }
println(commits)
val df = DiffFormatter(DisabledOutputStream.INSTANCE)
df.setRepository(git.repository)
df.setDetectRenames(true)
commits
    .filter { it.parents.isNotEmpty() && it.parents[0].tree != null }
    .forEach { commit ->
        println("parent:\t${commit.parents[0].name}")
        println("commit:\t${commit.name}")
        df.scan(commit.parents[0].tree, commit.tree).forEach { diff ->
            println(diff)
            val path = diff.newPath
            println(path)
            val editList = df.toFileHeader(diff).toEditList()
            for (edit in editList) {
                println(edit)
                println("A\t${edit.beginA} ${edit.endA}")
                println("B\t${edit.beginB} ${edit.endB}")
            }
    }
}


[commit de5e1f2a1421694a48860d0d1719f2186c8da8e1 1747297207 -----sp, commit fefa57d746a15b7130500baaf3ef5e61c56e744d 1747296430 -----sp, commit ccb523a266318fc4f2a18fc5fd2602ec5620dd56 1747296374 -----sp]
parent:	fefa57d746a15b7130500baaf3ef5e61c56e744d
commit:	de5e1f2a1421694a48860d0d1719f2186c8da8e1
DiffEntry[MODIFY SyncopeSyncResultHandler.java]
SyncopeSyncResultHandler.java
DELETE(244-247,244-244)
A	244 247
B	244 244
parent:	ccb523a266318fc4f2a18fc5fd2602ec5620dd56
commit:	fefa57d746a15b7130500baaf3ef5e61c56e744d
DiffEntry[MODIFY SyncopeSyncResultHandler.java]
SyncopeSyncResultHandler.java
DELETE(251-254,251-251)
A	251 254
B	251 251
